In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from basics import *
from clean_lc_paper_1 import *
plt.rcParams['text.usetex'] = True

In [ ]:
###everything from paper_1 version except for the high pass filter

In [ ]:
def calc_rms(vals):
    return np.sqrt(np.mean((vals-np.mean(vals))**2))

ds_bad_times = [131.43378, 181.60556, 230.40734, 255.43297, 256.19620, 258.60637, 280.65946, 281.18166, 290.66168,
                321.59224, 383.57387, 397.03067, 400.72203, 442.41804, 475.19640, 503.95780, 538.38311, 566.70265,
                598.47678, 629.44327, 660.45417, 690.46083, 711.14815, 761.27976, 802.52966, 845.14957, 874.27250,
                886.76523, 887.44811, 906.04662, 937.05752, 969.43418, 1032.4560, 1063.3865, 1070.0949, 1099.2981,
                1116.8522, 1118.6599, 1121.3914, 1122.0341, 1126.0511, 1154.0492, 1160.7576, 1182.1278, 1215.0626,
                1245.0291, 1273.1879, 1305.0424, 1336.4148, 1371.4828, 1404.4980, 1435.5089, 1471.2598, 1525.0870,
                1558.0662, 1559.1508, 1591.2463, 130.43898, 169.63898, 183.55898, 200.31608, 231.19898, 246.18889,
                260.11898, 291.31898, 323.43898, 349.63898, 352.23898, 372.75898, 384.15898, 443.31898, 476.15898,
                504.43898, 539.35898, 567.27898, 599.79898, 630.07898, 660.91898, 691.03898, 719.75898, 735.19898,
                763.59898, 808.39898, 845.55898, 874.63898, 906.71898, 937.71898, 969.99898, 1001.0390, 1032.9590,
                1064.0390, 1072.4390, 1098.4390, 1126.6390, 1154.7990, 1164.8790, 1182.5990, 1215.5990, 1245.7590,
                1274.0790, 1289.7190, 1296.1990, 1306.0390, 1337.1190, 1373.2390, 1405.1990, 1413.0390, 1418.3190,
                1436.1590, 1471.9590, 1477.5990, 1488.4390, 1525.9590, 1581.9190, 1586.6390]
ds_bad_times.sort()

In [ ]:
test_kic = 'kic-003735699'

In [ ]:
fname = test_kic
num_id = test_kic.split('-')[-1]

#open fits file and download lightcurve quarters
filepath = f'/Users/z5214708/tess_data/stello/Kepler/LC/KBonus/kbonus-bkg/lcs/{num_id[0:4]}/{num_id}/hlsp_kbonus-bkg_kepler_kepler_{fname}_kepler_v1.0_lc.fits'
stitched = Table.read(filepath, hdu=1, format='fits', unit_parse_strict='silent')


#remove NaNs
nan_mask = np.isnan(stitched['SAP_FLUX'])
time = np.asarray(stitched['TIME'][~nan_mask])
flux = np.asarray(stitched['SAP_FLUX'][~nan_mask])

min_time = min(time)
max_time = max(time)

#calculate power spectrum
org_freq, org_power = calc_lomb_scargle(time,flux)

#save original time and flux
org_time, org_flux = time.copy(), flux.copy()

In [ ]:
#Plot raw stitched flux
fig, axs = plt.subplots(2,1,figsize=(12, 12))
axs[0].scatter(time, flux, s=5,rasterized=True)
axs[0].set_xlabel('$\mathrm{Time\,\, (BKJD\,\, Days)}$',fontsize=25)
axs[0].set_ylabel('$\mathrm{Flux}$',fontsize=25)
axs[0].set_xlim(min(time), max(time))

axs[1].plot(org_freq,org_power)
axs[1].set_xscale('log')
axs[1].set_yscale('log')

for ax in axs:
    ax.tick_params(axis='both', which='major', labelsize=20, pad=5,size=8)

plt.show()

In [ ]:
# Split the light curve into stable segments
stable_times = []
stable_fluxes = []
for i in range(len(ds_bad_times) - 1):
    start_time = ds_bad_times[i]
    end_time = ds_bad_times[i + 1]

    # Extract data within the current stable segment
    segment_mask = (time > start_time) & (time <= end_time)
    segment_time = time[segment_mask]
    segment_flux = flux[segment_mask]

    if len(segment_flux) > 0:  # Skip empty segments
        segment_length = max(segment_time) - min(segment_time)
        if segment_length > 1:
            stable_times.append(segment_time)
            stable_fluxes.append(segment_flux)

In [ ]:
#plot, colour coding by stable segments
plt.figure(figsize=(12, 6))
for i in range(len(stable_times)):
    plt.scatter(stable_times[i], stable_fluxes[i], 5, rasterized = True)
plt.text(0.02, 0.98, '$\mathrm{b)}$', 
         transform=plt.gca().transAxes,
         fontsize=25,
         verticalalignment='top',
         horizontalalignment='left')   
plt.xlim(min_time,max_time)
plt.xlabel('$\mathrm{Time\,\, (BKJD\,\, Days)}$',fontsize=25)
plt.ylabel('$\mathrm{Flux}$',fontsize=25)
plt.tick_params(axis='both', which='major', labelsize=20, pad=5,size=8)
plt.show()

In [ ]:
#normalize and sigma clip
for i in range(len(stable_times)):
    ptp = calc_ptp(stable_fluxes[i])
    
    rms_ptp = np.sqrt(np.mean((ptp-np.mean(ptp))**2))
    
    #5 sigma clip based on ptp array
    threshold = 5 * np.std(ptp)
    outliers_mask = np.abs(ptp) >= threshold

    stable_times[i] = stable_times[i][~outliers_mask]
    stable_fluxes[i] = stable_fluxes[i][~outliers_mask]
    
    #Normalize
    stable_fluxes[i] = stable_fluxes[i] / np.median(stable_fluxes[i])

In [ ]:
#plot
plt.figure(figsize=(12, 6))
for i in range(len(stable_times)):
    plt.scatter(stable_times[i], stable_fluxes[i], s=5, rasterized=True)
plt.text(0.02, 0.98, '$\mathrm{c)}$', 
         transform=plt.gca().transAxes,
         fontsize=25,
         verticalalignment='top',
         horizontalalignment='left')   
plt.xlim(min_time,max_time)
plt.xlabel('$\mathrm{Time\,\, (BKJD\,\, Days)}$',fontsize=25)
plt.ylabel('$\mathrm{Normalised \,\,Flux}$',fontsize=25)
plt.tick_params(axis='both', which='major', labelsize=20, pad=5,size=8)
plt.savefig('clean_lc_c.pdf',format='pdf',bbox_inches='tight')

In [ ]:
#Concatenate quarters
time = np.concatenate(stable_times)
flux = np.concatenate(stable_fluxes)

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(time, flux, s=5, rasterized=True)
plt.text(0.02, 0.98, '$\mathrm{e)}$', 
         transform=plt.gca().transAxes,
         fontsize=25,
         verticalalignment='top',
         horizontalalignment='left')   
plt.xlim(min_time,max_time)
plt.xlabel('$\mathrm{Time\,\, (BKJD\,\, Days)}$',fontsize=25)
plt.ylabel('$\mathrm{Normalised \,\,Flux}$',fontsize=25)
plt.tick_params(axis='both', which='major', labelsize=20, pad=5,size=8)
plt.show()

In [ ]:
cutoff = 2
#Split LC into quarters and cut quarters where std is >1.3 x the minimum quarter std
time_quarters, flux_quarters = split_light_curve(time.copy(), flux.copy())
time_quarters, flux_quarters = cut_quarters(time_quarters, flux_quarters, cutoff)

#Fill single cadence gaps
time = np.concatenate(time_quarters)
flux = np.concatenate(flux_quarters)
time, flux = fill_single_cadence_gaps(time, flux)

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(time, flux, 5)
plt.ylim(0.993,1.008)
plt.text(0.02, 0.98, '$\mathrm{f)}$', 
         transform=plt.gca().transAxes,
         fontsize=25,
         verticalalignment='top',
         horizontalalignment='left')   
plt.xlim(min_time,max_time)
plt.xlabel('$\mathrm{Time\,\, (BKJD\,\, Days)}$',fontsize=25,rasterized=True)
plt.ylabel('$\mathrm{Normalised \,\,Flux}$',fontsize=25)
plt.tick_params(axis='both', which='major', labelsize=20, pad=5,size=8)
plt.show()

In [ ]:
#calc frequency and power after all that
final_freq, final_power = calc_lomb_scargle(time,flux)

#also save final time and flux
final_time, final_flux = time.copy(), flux.copy()

In [ ]:
#plot
plt.figure(figsize=(12, 6))
plt.plot(final_freq,final_power)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('$\mathrm{Frequency\,\, [\mu Hz]}$',fontsize=25)
plt.ylabel('$\mathrm{Power\,\,[ppm^2/ \mu Hz]}$',fontsize=25)
plt.tick_params(axis='both', which='major', labelsize=20, pad=5,size=8)

In [ ]:
# plot 2x3: rows = light curve, power spectrum
# columns = original, cleaned, direct comparison
fig, axs = plt.subplots(2, 3, figsize=(18, 12), sharey='row')

# --- normalized data ---
org_flux_norm = org_flux / np.median(org_flux)
org_power_norm = org_power / np.median(org_power)
final_power_norm = final_power / np.median(final_power)

# --- shared axis ranges ---
# light curve row
lc_xmin = min(np.min(org_time), np.min(final_time))
lc_xmax = max(np.max(org_time), np.max(final_time))
lc_ymin = min(np.min(org_flux_norm), np.min(final_flux))
lc_ymax = max(np.max(org_flux_norm), np.max(final_flux))

# power spectrum row
ps_xmin = min(np.min(org_freq), np.min(final_freq))
ps_xmax = max(np.max(org_freq), np.max(final_freq))
ps_ymin = min(np.min(org_power_norm), np.min(final_power_norm))
ps_ymax = max(np.max(org_power_norm), np.max(final_power_norm))

# --- Row 0: light curve ---
axs[0, 0].scatter(org_time, org_flux_norm, s=1, c='darkblue')
axs[0, 0].set_title(r'$\mathrm{Raw}$', fontsize=30)
axs[0, 0].set_xlabel(r'$\mathrm{Time\,\, [Days]}$', fontsize=25)
axs[0, 0].set_ylabel(r'$\mathrm{Normalized\,\, Flux}$', fontsize=25)

axs[0, 1].scatter(final_time, final_flux, s=1, c='red')
axs[0, 1].set_title(r'$\mathrm{Cleaned}$', fontsize=30)
axs[0, 1].set_xlabel(r'$\mathrm{Time\,\, [Days]}$', fontsize=25)

axs[0, 2].scatter(org_time, org_flux_norm, s=1, c='darkblue', label='org')
axs[0, 2].scatter(final_time, final_flux, s=1, c='red', alpha=0.5, label='final')
axs[0, 2].set_title(r'$\mathrm{Comparison}$', fontsize=30)
axs[0, 2].set_xlabel(r'$\mathrm{Time\,\, [Days]}$', fontsize=25)

for ax in axs[0, :]:
    ax.set_xlim(lc_xmin, lc_xmax)
    ax.set_ylim(lc_ymin, lc_ymax)

# --- Row 1: power spectrum ---
axs[1, 0].plot(org_freq, org_power_norm, c='darkblue')
axs[1, 0].set_xlabel(r'$\mathrm{Frequency\,\,[\mu Hz]}$', fontsize=25)
axs[1, 0].set_ylabel(r'$\mathrm{Normalized\,\, Power}$', fontsize=25)

axs[1, 1].plot(final_freq, final_power_norm, c='red')
axs[1, 1].set_xlabel(r'$\mathrm{Frequency\,\,[\mu Hz]}$', fontsize=25)

axs[1, 2].plot(org_freq, org_power_norm, c='darkblue', label='org')
axs[1, 2].plot(final_freq, final_power_norm, c='red', alpha=0.8, label='final')
axs[1, 2].set_xlabel(r'$\mathrm{Frequency\,\,[\mu Hz]}$', fontsize=25)

for ax in axs[1, :]:
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(ps_xmin, ps_xmax)
    ax.set_ylim(ps_ymin, ps_ymax)

# turn off y tick labels for all but the leftmost column
for ax in axs[:, 1:].flat:
    ax.tick_params(axis='y', labelleft=False)

# increase tick label size across all subplots
for ax in axs.flat:
    ax.tick_params(axis='both', labelsize=20)

plt.subplots_adjust(wspace=0)
plt.savefig('../figures/cleaning_test_kic-003735699.png',bbox_inches='tight')

In [ ]:
def clean_lc_v1(time,flux, ds_bad_times):
    ###split into stable time segments, remove NaNs, sigma clip, then remove dodgy quarters as in paper_1
    stable_times = []
    stable_fluxes = []
    for i in range(len(ds_bad_times) - 1):
        start_time = ds_bad_times[i]
        end_time = ds_bad_times[i + 1]

        # Extract data within the current stable segment
        segment_mask = (time > start_time) & (time <= end_time)
        segment_time = time[segment_mask]
        segment_flux = flux[segment_mask]

        if len(segment_flux) > 0:  # Skip empty segments
            segment_length = max(segment_time) - min(segment_time)
            if segment_length > 1:
                stable_times.append(segment_time)
                stable_fluxes.append(segment_flux)

    #normalize and sigma clip
    for i in range(len(stable_times)):
        ptp = calc_ptp(stable_fluxes[i])
        
        rms_ptp = np.sqrt(np.mean((ptp-np.mean(ptp))**2))
        
        #5 sigma clip based on ptp array
        threshold = 5 * np.std(ptp)
        outliers_mask = np.abs(ptp) >= threshold

        stable_times[i] = stable_times[i][~outliers_mask]
        stable_fluxes[i] = stable_fluxes[i][~outliers_mask]
        
        #Normalize
        stable_fluxes[i] = stable_fluxes[i] / np.median(stable_fluxes[i])

    #Concatenate quarters
    time = np.concatenate(stable_times)
    flux = np.concatenate(stable_fluxes)

    cutoff = 2
    #Split LC into quarters and cut quarters where std is >1.3 x the minimum quarter std
    time_quarters, flux_quarters = split_light_curve(time.copy(), flux.copy())
    time_quarters, flux_quarters = cut_quarters(time_quarters, flux_quarters, cutoff)

    #Fill single cadence gaps
    time = np.concatenate(time_quarters)
    flux = np.concatenate(flux_quarters)
    time, flux = fill_single_cadence_gaps(time, flux)

    return time, flux

In [ ]:
def plot_lc_comparison(org_time,org_flux,final_time,final_flux,org_freq,org_power,final_freq,final_power,num_id,folder,category):

    # plot 2x3: rows = light curve, power spectrum
    # columns = original, cleaned, direct comparison
    fig, axs = plt.subplots(2, 3, figsize=(18, 12), sharey='row')

    # --- normalized data ---
    org_flux_norm = org_flux / np.median(org_flux)
    org_power_norm = org_power / np.median(org_power)
    final_power_norm = final_power / np.median(final_power)

    # --- shared axis ranges ---
    # light curve row
    lc_xmin = min(np.min(org_time), np.min(final_time))
    lc_xmax = max(np.max(org_time), np.max(final_time))
    lc_ymin = min(np.min(org_flux_norm), np.min(final_flux))
    lc_ymax = max(np.max(org_flux_norm), np.max(final_flux))

    # power spectrum row
    ps_xmin = min(np.min(org_freq), np.min(final_freq))
    ps_xmax = max(np.max(org_freq), np.max(final_freq))
    ps_ymin = min(np.min(org_power_norm), np.min(final_power_norm))
    ps_ymax = max(np.max(org_power_norm), np.max(final_power_norm))

    # --- Row 0: light curve ---
    axs[0, 0].scatter(org_time, org_flux_norm, s=1, c='darkblue')
    axs[0, 0].set_title(r'$\mathrm{Raw}$', fontsize=30)
    axs[0, 0].set_xlabel(r'$\mathrm{Time\,\, [Days]}$', fontsize=25)
    axs[0, 0].set_ylabel(r'$\mathrm{Normalized\,\, Flux}$', fontsize=25)

    axs[0, 1].scatter(final_time, final_flux, s=1, c='red')
    axs[0, 1].set_title(r'$\mathrm{Cleaned}$', fontsize=30)
    axs[0, 1].set_xlabel(r'$\mathrm{Time\,\, [Days]}$', fontsize=25)

    axs[0, 2].scatter(org_time, org_flux_norm, s=1, c='darkblue', label='org')
    axs[0, 2].scatter(final_time, final_flux, s=1, c='red', alpha=0.5, label='final')
    axs[0, 2].set_title(r'$\mathrm{Comparison}$', fontsize=30)
    axs[0, 2].set_xlabel(r'$\mathrm{Time\,\, [Days]}$', fontsize=25)

    for ax in axs[0, :]:
        ax.set_xlim(lc_xmin, lc_xmax)
        ax.set_ylim(lc_ymin, lc_ymax)

    # --- Row 1: power spectrum ---
    axs[1, 0].plot(org_freq, org_power_norm, c='darkblue')
    axs[1, 0].set_xlabel(r'$\mathrm{Frequency\,\,[\mu Hz]}$', fontsize=25)
    axs[1, 0].set_ylabel(r'$\mathrm{Normalized\,\, Power}$', fontsize=25)

    axs[1, 1].plot(final_freq, final_power_norm, c='red')
    axs[1, 1].set_xlabel(r'$\mathrm{Frequency\,\,[\mu Hz]}$', fontsize=25)

    axs[1, 2].plot(org_freq, org_power_norm, c='darkblue', label='org')
    axs[1, 2].plot(final_freq, final_power_norm, c='red', alpha=0.8, label='final')
    axs[1, 2].set_xlabel(r'$\mathrm{Frequency\,\,[\mu Hz]}$', fontsize=25)

    for ax in axs[1, :]:
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlim(ps_xmin, ps_xmax)
        ax.set_ylim(ps_ymin, ps_ymax)

    # turn off y tick labels for all but the leftmost column
    for ax in axs[:, 1:].flat:
        ax.tick_params(axis='y', labelleft=False)

    # increase tick label size across all subplots
    for ax in axs.flat:
        ax.tick_params(axis='both', labelsize=20)

    plt.subplots_adjust(wspace=0)
    plt.savefig(f'../figures/{folder}/{category}/{num_id}_cleaning_v1_comparison.png',bbox_inches='tight')
    plt.close()

In [ ]:
###try it for a bunch
sample_bright = pd.read_csv('../data/sample_bright.csv')
sample_mid = pd.read_csv('../data/sample_mid.csv')
sample_faint = pd.read_csv('../data/sample_faint.csv')

for df, folder in zip([sample_bright,sample_mid,sample_faint],['sample_bright','sample_mid','sample_faint']):
    for i in range(len(df)):
        fname = df['fname'].iloc[i]
        num_id = fname.split('-')[-1]
        category = df['category'].iloc[i]

        #open fits file and download lightcurve quarters
        filepath = f'/Users/z5214708/tess_data/stello/Kepler/LC/KBonus/kbonus-bkg/lcs/{num_id[0:4]}/{num_id}/hlsp_kbonus-bkg_kepler_kepler_{fname}_kepler_v1.0_lc.fits'
        stitched = Table.read(filepath, hdu=1, format='fits', unit_parse_strict='silent')


        #remove NaNs
        nan_mask = np.isnan(stitched['SAP_FLUX'])
        time = np.asarray(stitched['TIME'][~nan_mask])
        flux = np.asarray(stitched['SAP_FLUX'][~nan_mask])

        if len(time) > 1500:
            #calculate power spectrum
            org_freq, org_power = calc_lomb_scargle(time,flux)

            #save original time and flux
            org_time, org_flux = time.copy(), flux.copy()

            final_time, final_flux = clean_lc_v1(org_time,org_flux,ds_bad_times)
            final_freq, final_power = calc_lomb_scargle(final_time,final_flux)

            plot_lc_comparison(org_time,org_flux,final_time,final_flux,org_freq,org_power,final_freq,final_power,num_id,folder,category)
        
        

In [ ]:
for df in 